# OLAP Transformation Testing

### Validate incremental dimension, bridge, and fact loading before Airflow orchestration.

In [1]:
# Import sys so the notebook can load project modules.
import sys

# Import Path for safe project-path handling.
from pathlib import Path


# The notebook lives inside /notebooks, so the project root is one level above.
project_root = Path.cwd().parent


if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Import the production database connection helper.
from src.database import get_etl_connection


print("Project root:", project_root)

Project root: /Users/mac/Documents/netflix-data-engineering


## Reusable helper for multi-statement SQL

In [12]:
# Execute a production SQL script containing one or more SQL statements.
def execute_sql_script(sql_text, parameters=None):

    # Split simple SQL scripts into individual statements.
    statements = [
        statement.strip()
        for statement in sql_text.split(";")
        if statement.strip()
    ]

    # Execute all statements inside one database transaction.
    with get_etl_connection() as connection:
        with connection.cursor() as cursor:

            for statement in statements:
                cursor.execute(
                    statement,
                    parameters or {},
                )

        # Commit only after every statement succeeds.
        connection.commit()

## Next get the latest titles batch

In [2]:
# Find the latest titles batch already registered by the pipeline.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT batch_id
            FROM etl.batch_history
            WHERE file_name = 'titles.csv'
            ORDER BY batch_id DESC
            LIMIT 1;
            """
        )

        latest_titles_batch = cursor.fetchone()


# Confirm a titles batch exists.
assert latest_titles_batch is not None


# Extract the batch ID.
titles_batch_id = latest_titles_batch[0]


print("Titles batch ID:", titles_batch_id)

Titles batch ID: 12


## Run the dimension upsert

In [3]:
# Locate the production dim_title SQL script.
dim_title_sql_path = (
    project_root
    / "sql"
    / "03_olap"
    / "01_upsert_dim_title.sql"
)

# Confirm the SQL file exists.
assert dim_title_sql_path.exists()

# Read the production SQL.
dim_title_sql = dim_title_sql_path.read_text(
    encoding="utf-8"
)


# Count the dimension before the transformation.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.dim_title;
            """
        )

        dim_title_count_before = cursor.fetchone()[0]


# Run the incremental dimension upsert.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            dim_title_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    connection.commit()


# Count the dimension after the transformation.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.dim_title;
            """
        )

        dim_title_count_after = cursor.fetchone()[0]


print("dim_title before:", dim_title_count_before)
print("dim_title after: ", dim_title_count_after)

print("PASS: dim_title incremental upsert completed.")

dim_title before: 5849
dim_title after:  5849
PASS: dim_title incremental upsert completed.


## Test the Remaining dimension upserts

Validate incremental loading of person, genre, and country dimensions.

In [4]:
# Run and validate a dimension SQL script.
def test_dimension_upsert(
    sql_filename,
    table_name,
):
    # Build the production SQL path.
    sql_path = (
        project_root
        / "sql"
        / "03_olap"
        / sql_filename
    )

    assert sql_path.exists()

    # Load the SQL script.
    sql_script = sql_path.read_text(
        encoding="utf-8"
    )

    # Count rows before execution.
    with get_etl_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                f"""
                SELECT COUNT(*)
                FROM analytics.{table_name};
                """
            )

            count_before = cursor.fetchone()[0]

    # Run the production transformation.
    with get_etl_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(sql_script)

        connection.commit()

    # Count rows after execution.
    with get_etl_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                f"""
                SELECT COUNT(*)
                FROM analytics.{table_name};
                """
            )

            count_after = cursor.fetchone()[0]

    print(f"{table_name} before:", count_before)
    print(f"{table_name} after: ", count_after)

    return sql_script, count_after

## Test for dim_person

In [5]:
person_sql, person_count_after = test_dimension_upsert(
    "02_upsert_dim_person.sql",
    "dim_person",
)

dim_person before: 54589
dim_person after:  54589


## Test for dim_genre

In [6]:
genre_sql, genre_count_after = test_dimension_upsert(
    "03_upsert_dim_genre.sql",
    "dim_genre",
)

dim_genre before: 19
dim_genre after:  19


## Test for dim_country

In [7]:
country_sql, country_count_after = test_dimension_upsert(
    "04_upsert_dim_country.sql",
    "dim_country",
)

dim_country before: 109
dim_country after:  109


## prove idempotency for dim_genre, dim_person, dim_country

In [8]:
# Re-run all three transformations.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(person_sql)
        cursor.execute(genre_sql)
        cursor.execute(country_sql)

    connection.commit()


# Verify row counts remain unchanged.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute(
            "SELECT COUNT(*) FROM analytics.dim_person;"
        )
        person_count_rerun = cursor.fetchone()[0]

        cursor.execute(
            "SELECT COUNT(*) FROM analytics.dim_genre;"
        )
        genre_count_rerun = cursor.fetchone()[0]

        cursor.execute(
            "SELECT COUNT(*) FROM analytics.dim_country;"
        )
        country_count_rerun = cursor.fetchone()[0]


assert person_count_rerun == person_count_after
assert genre_count_rerun == genre_count_after
assert country_count_rerun == country_count_after

print("PASS: remaining dimension upserts are idempotent.")

PASS: remaining dimension upserts are idempotent.


## Title-genre bridge synchronisation

Synchronise OLTP title/genre relationships into the analytics bridge using stable surrogate keys.

In [13]:
# Locate the title-genre bridge SQL.
bridge_genre_sql_path = (
    project_root
    / "sql"
    / "03_olap"
    / "05_sync_bridge_title_genre.sql"
)

assert bridge_genre_sql_path.exists()

# Read the production SQL.
bridge_genre_sql = bridge_genre_sql_path.read_text(
    encoding="utf-8"
)


# Capture the current bridge count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_genre;
            """
        )

        bridge_genre_count_before = cursor.fetchone()[0]


# Execute DELETE + INSERT in one transaction.
execute_sql_script(
    bridge_genre_sql,
    {
        "batch_id": titles_batch_id,
    },
)


# Capture the resulting count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_genre;
            """
        )

        bridge_genre_count_after = cursor.fetchone()[0]


print("Genre bridge before:", bridge_genre_count_before)
print("Genre bridge after: ", bridge_genre_count_after)

print("PASS: title-genre bridge synchronised.")

Genre bridge before: 15088
Genre bridge after:  15088
PASS: title-genre bridge synchronised.


## Prove idempotency

In [14]:
# Locate the title-genre bridge SQL.
bridge_genre_sql_path = (
    project_root
    / "sql"
    / "03_olap"
    / "05_sync_bridge_title_genre.sql"
)

assert bridge_genre_sql_path.exists()

# Read the production SQL.
bridge_genre_sql = bridge_genre_sql_path.read_text(
    encoding="utf-8"
)


# Capture the current bridge count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_genre;
            """
        )

        bridge_genre_count_before = cursor.fetchone()[0]


# Execute DELETE + INSERT in one transaction.
execute_sql_script(
    bridge_genre_sql,
    {
        "batch_id": titles_batch_id,
    },
)


# Capture the resulting count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_genre;
            """
        )

        bridge_genre_count_after = cursor.fetchone()[0]


print("Genre bridge before:", bridge_genre_count_before)
print("Genre bridge after: ", bridge_genre_count_after)

print("PASS: title-genre bridge synchronised.")

Genre bridge before: 15088
Genre bridge after:  15088
PASS: title-genre bridge synchronised.


## Title-country bridge synchronisation

Synchronise OLTP title/country relationships into the analytics bridge using stable surrogate keys.

In [15]:
# Locate the title-country bridge SQL.
bridge_country_sql_path = (
    project_root
    / "sql"
    / "03_olap"
    / "06_sync_bridge_title_country.sql"
)

assert bridge_country_sql_path.exists()

# Read the production SQL.
bridge_country_sql = bridge_country_sql_path.read_text(
    encoding="utf-8"
)


# Capture the current bridge count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_country;
            """
        )

        bridge_country_count_before = cursor.fetchone()[0]


# Execute DELETE + INSERT in one transaction.
execute_sql_script(
    bridge_country_sql,
    {
        "batch_id": titles_batch_id,
    },
)


# Capture the resulting bridge count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_country;
            """
        )

        bridge_country_count_after = cursor.fetchone()[0]


print("Country bridge before:", bridge_country_count_before)
print("Country bridge after: ", bridge_country_count_after)

print("PASS: title-country bridge synchronised.")

Country bridge before: 6528
Country bridge after:  6528
PASS: title-country bridge synchronised.


## Keep the bridge integrity test

In [16]:
# Verify that every bridge key still resolves to a valid dimension row.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_genre AS bridge

            LEFT JOIN analytics.dim_title AS dt
                ON dt.title_sk = bridge.title_sk

            LEFT JOIN analytics.dim_genre AS dg
                ON dg.genre_sk = bridge.genre_sk

            WHERE dt.title_sk IS NULL
               OR dg.genre_sk IS NULL;
            """
        )

        orphan_genre_bridges = cursor.fetchone()[0]


        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.bridge_title_country AS bridge

            LEFT JOIN analytics.dim_title AS dt
                ON dt.title_sk = bridge.title_sk

            LEFT JOIN analytics.dim_country AS dc
                ON dc.country_sk = bridge.country_sk

            WHERE dt.title_sk IS NULL
               OR dc.country_sk IS NULL;
            """
        )

        orphan_country_bridges = cursor.fetchone()[0]


print("Orphan genre relationships:", orphan_genre_bridges)
print("Orphan country relationships:", orphan_country_bridges)

assert orphan_genre_bridges == 0
assert orphan_country_bridges == 0

print("PASS: OLAP bridge integrity checks passed.")

Orphan genre relationships: 0
Orphan country relationships: 0
PASS: OLAP bridge integrity checks passed.


## Test fact_credit incremental loading

In [17]:
# ============================================================
# Credit fact incremental load test
# ============================================================

from pathlib import Path

fact_credit_sql_path = (
    project_root
    / "sql"
    / "03_olap"
    / "07_upsert_fact_credits.sql"
)

fact_credit_sql = fact_credit_sql_path.read_text()


with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        # Count before.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.fact_credits;
            """
        )

        fact_credit_count_before = cursor.fetchone()[0]

        # Run fact load.
        cursor.execute(fact_credit_sql)

        connection.commit()

        # Count after.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.fact_credits;
            """
        )

        fact_credit_count_after = cursor.fetchone()[0]


print("fact_credits before:", fact_credit_count_before)
print("fact_credits after:", fact_credit_count_after)

print("PASS: fact_credits load completed.")

fact_credits before: 77800
fact_credits after: 77800
PASS: fact_credits load completed.


## Validate fact_credits completeness and referential integrity

Verify that every valid OLTP credit that can be mapped through the
dimensions exists in the OLAP fact table, and ensure that the fact
table contains no orphan surrogate keys.

In [18]:
# ============================================================
# Validate fact_credits completeness
# ============================================================

with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        # Number of valid logical credits in OLTP that can be
        # resolved through both OLAP dimensions.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM (
                SELECT DISTINCT
                    dt.title_sk,
                    dp.person_sk,
                    c.role,
                    c.character

                FROM public.credit AS c

                INNER JOIN analytics.dim_title AS dt
                    ON dt.title_id = c.title_id

                INNER JOIN analytics.dim_person AS dp
                    ON dp.person_id = c.person_id

                WHERE c.title_id IS NOT NULL
                  AND c.person_id IS NOT NULL
                  AND c.role IS NOT NULL
            ) AS expected_credits;
            """
        )

        expected_fact_credits = cursor.fetchone()[0]


        # Actual OLAP fact count.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.fact_credits;
            """
        )

        actual_fact_credits = cursor.fetchone()[0]


print("Expected mapped credits:", expected_fact_credits)
print("Actual fact_credits:    ", actual_fact_credits)

assert expected_fact_credits == actual_fact_credits, (
    "fact_credits count does not match the expected OLTP relationships."
)

print("PASS: fact_credits completeness check passed.")

Expected mapped credits: 77800
Actual fact_credits:     77800
PASS: fact_credits completeness check passed.


## Test for orphan foreign keys

In [19]:
# ============================================================
# Validate fact_credits referential integrity
# ============================================================

with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT COUNT(*)
            FROM analytics.fact_credits AS fc

            LEFT JOIN analytics.dim_title AS dt
                ON dt.title_sk = fc.title_sk

            LEFT JOIN analytics.dim_person AS dp
                ON dp.person_sk = fc.person_sk

            WHERE dt.title_sk IS NULL
               OR dp.person_sk IS NULL;
            """
        )

        orphan_fact_credits = cursor.fetchone()[0]


print("Orphan fact credit relationships:", orphan_fact_credits)

assert orphan_fact_credits == 0, (
    "fact_credits contains orphan dimension relationships."
)

print("PASS: fact_credits referential integrity passed.")

Orphan fact credit relationships: 0
PASS: fact_credits referential integrity passed.
